<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">05 Mobile AI</h1>
<hr>


## **3. Understanding Attention**

<div class="alert alert-light">

  **Name:** (add your name here)
  
  **Matriculation Number:** (add your number here)
  
</div>

### 3.1 Introduction

This notebook explains the Query, Key, and Value matrices used inside transformer attention. To keep the example consistent and easy to inspect, we use four short sentences: three facts and one question. Each complete sentence is represented by one small feature vector. We then project those vectors into queries, keys, and values, compute attention scores, normalize the scores into attention weights, and combine the value vectors into contextualized outputs.

Real transformers normally apply attention to token representations rather than to complete sentences. The sentence-level simplification used here keeps the matrices small while preserving the same attention calculation.

In [ ]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)

sentences = [
    "Zurich is awesome in summer.",
    "Bern is cool in summer.",
    "Lausanne has a nice view in summer.",
    "How is Zurich in summer?",
]

sentence_labels = ["Zurich fact", "Bern fact", "Lausanne fact", "Zurich question"]
feature_names = [
    "Zurich", "Bern", "Lausanne", "summer",
    "awesome", "cool", "nice view", "question",
]

# Tiny, human-readable sentence representations: 4 sentences, 8 features each.
# Real models learn dense representations with many more dimensions.
X = np.array([
    [1, 0, 0, 1, 1, 0, 0, 0],  # Zurich fact
    [0, 1, 0, 1, 0, 1, 0, 0],  # Bern fact
    [0, 0, 1, 1, 0, 0, 1, 0],  # Lausanne fact
    [1, 0, 0, 1, 0, 0, 0, 1],  # Zurich question
])

for label, sentence in zip(sentence_labels, sentences):
    print(f"{label:17s}: {sentence}")

print("\nFeature order:", feature_names)
print("\nInput representation matrix X:")
print(X)
print("Shape:", X.shape)

### 3.2 Projection Matrices

Attention lets each sentence decide which other sentences are relevant when building its next representation. For every sentence, the model creates three vectors:

- `Q`, the query vector: the representation used to compare this sentence against other sentences.
- `K`, the key vector: the representation other sentences compare against.
- `V`, the value vector: the content that will be mixed into the output.

The attention pattern comes from comparing queries with keys. The output comes from using those attention weights to blend values. This split gives the model separate learned spaces for matching and for carrying information forward.

$Q$, $K$, and $V$ are not themselves learnable model parameters. They are intermediate results that the model computes from the input matrix $X$ during each forward pass. The learnable parameters are the projection matrices $\mathbf{W}_Q$, $\mathbf{W}_K$, and $\mathbf{W}_V$.

Each projection matrix transforms the same input embeddings into a different role-specific space:

- $Q = X\mathbf{W}_Q$ produces the query vectors used to ask what each sentence is looking for.
- $K = X\mathbf{W}_K$ produces the key vectors used to describe what each sentence offers for matching.
- $V = X\mathbf{W}_V$ produces the value vectors containing the information that can be passed forward.

At the beginning of training, the entries of $\mathbf{W}_Q$, $\mathbf{W}_K$, and $\mathbf{W}_V$ are normally initialized with small random values. During training, the model performs a forward pass, measures its prediction error with a loss function, and uses backpropagation to compute gradients for these entries. An optimizer then updates the projection matrices so that the resulting queries, keys, and values become more useful for the model's task.

For the numerical walkthrough, we use fixed, easy-to-read projection matrices that behave like a useful learned solution. They make it possible to inspect exactly how the Zurich question finds the Zurich fact and retrieves the description `awesome`. Section 3.3 temporarily switches to a separate trainable model to show how gradient descent can learn this behavior. Sections 3.4–3.7 then return to the transparent fixed matrices so that every attention calculation remains easy to follow.

In [ ]:
d_model = X.shape[1]
d_k = 4
d_v = 3

# W_Q and W_K use city and season features for matching.
# The factor 2 makes the useful matches easy to see after softmax.
W_Q = np.array([
    [2, 0, 0, 0],  # Zurich
    [0, 2, 0, 0],  # Bern
    [0, 0, 2, 0],  # Lausanne
    [0, 0, 0, 2],  # summer
    [0, 0, 0, 0],  # awesome
    [0, 0, 0, 0],  # cool
    [0, 0, 0, 0],  # nice view
    [0, 0, 0, 0],  # question
], dtype=float)

W_K = W_Q.copy()
# Keep the question from treating itself as another factual answer source.
W_K[7] = [-2, -2, -2, -2]

# W_V extracts the descriptive information carried by each factual sentence.
W_V = np.array([
    [0, 0, 0],  # Zurich
    [0, 0, 0],  # Bern
    [0, 0, 0],  # Lausanne
    [0, 0, 0],  # summer
    [1, 0, 0],  # awesome
    [0, 1, 0],  # cool
    [0, 0, 1],  # nice view
    [0, 0, 0],  # question
], dtype=float)

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

print("W_Q shape:", W_Q.shape, "-> Q shape:", Q.shape)
print("W_K shape:", W_K.shape, "-> K shape:", K.shape)
print("W_V shape:", W_V.shape, "-> V shape:", V.shape)
print("\nQ:\n", Q)
print("\nK:\n", K)
print("\nV:\n", V)

### 3.3 Learning attention weights with gradient descent

Section 3.2 used transparent projection matrices chosen by hand. We now create a separate tiny model whose projection matrices begin with random values and are learned from examples.

We keep the same four-sentence prompt:

```text
Zurich is awesome in summer.
Bern is cool in summer.
Lausanne has a nice view in summer.
How is Zurich in summer?
```

The correct answer is `awesome`. The model must learn that the question matches the first fact because both mention Zurich and summer. The other facts mention the same season but different cities.

For every sentence vector $x_i$, the model computes a query, key, and value:

$$
q = x_{\text{question}}W_Q, \qquad k_i = x_iW_K, \qquad v_i = x_iW_V
$$

The question query is compared with all sentence keys. Softmax turns the resulting scores into attention weights, and those weights mix the value vectors into a context vector. A small classifier uses that context vector to predict a description such as `awesome`, `cool`, or `nice view`.

The model receives only the correct description as its training label. It is not told which sentence contains that description. Gradient descent must therefore make the query, key, and value projections useful for the final prediction:

$$
\text{random parameters} \rightarrow \text{prediction} \rightarrow \text{loss} \rightarrow
\text{parameter update}
$$

The following three figures show the model before training, its progress during gradient descent, and the same fixed prompt after training. Later sections return to the fixed NumPy matrices from Section 3.2 so that every attention calculation remains easy to inspect.

#### 3.3.1 Prediction before training

The model below starts with random parameters. The question sentence cannot attend to itself, but the model does not yet know which of the three factual sentences is useful. The first graph therefore shows an unreliable attention pattern and prediction.

In [ ]:
import math
import random
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

torch.manual_seed(11)
np.random.seed(11)
random.seed(11)
torch.set_num_threads(1)

TRAIN_CITIES = ["Zurich", "Bern", "Lausanne", "Basel", "Geneva", "Lucerne"]
TRAIN_SEASONS = ["spring", "summer", "autumn", "winter"]
TRAIN_DESCRIPTIONS = ["awesome", "cool", "nice view", "quiet", "lively", "peaceful"]

TRAIN_FEATURE_NAMES = (
    [f"city={city}" for city in TRAIN_CITIES]
    + [f"season={season}" for season in TRAIN_SEASONS]
    + [f"description={description}" for description in TRAIN_DESCRIPTIONS]
    + ["is_fact", "is_question"]
)

CITY_OFFSET = 0
SEASON_OFFSET = len(TRAIN_CITIES)
DESCRIPTION_OFFSET = len(TRAIN_CITIES) + len(TRAIN_SEASONS)
TYPE_OFFSET = DESCRIPTION_OFFSET + len(TRAIN_DESCRIPTIONS)
TRAIN_D_IN = len(TRAIN_FEATURE_NAMES)

def training_fact_sentence(city, season, description):
    if description == "nice view":
        return f"{city} has a nice view in {season}."
    return f"{city} is {description} in {season}."

def training_question_sentence(city, season):
    return f"How is {city} in {season}?"

def training_sentence_vector(kind, city, season, description=None):
    vector = np.zeros(TRAIN_D_IN, dtype=np.float32)
    vector[CITY_OFFSET + TRAIN_CITIES.index(city)] = 1.0
    vector[SEASON_OFFSET + TRAIN_SEASONS.index(season)] = 1.0
    if kind == "fact":
        vector[DESCRIPTION_OFFSET + TRAIN_DESCRIPTIONS.index(description)] = 1.0
        vector[TYPE_OFFSET] = 1.0
    else:
        vector[TYPE_OFFSET + 1] = 1.0
    return vector

def make_training_example():
    """Create three facts and one question with two hard distractors."""
    target_city = random.choice(TRAIN_CITIES)
    target_season = random.choice(TRAIN_SEASONS)
    target_description = random.choice(TRAIN_DESCRIPTIONS)

    wrong_season = random.choice([
        season for season in TRAIN_SEASONS if season != target_season
    ])
    wrong_city = random.choice([
        city for city in TRAIN_CITIES if city != target_city
    ])
    distractor_descriptions = random.sample([
        description
        for description in TRAIN_DESCRIPTIONS
        if description != target_description
    ], 2)

    correct_fact = (target_city, target_season, target_description)
    same_city_fact = (target_city, wrong_season, distractor_descriptions[0])
    same_season_fact = (wrong_city, target_season, distractor_descriptions[1])
    facts = [correct_fact, same_city_fact, same_season_fact]
    random.shuffle(facts)

    question_index = random.randrange(4)
    fact_iterator = iter(facts)
    sentence_texts = []
    sentence_vectors = []
    answer_sentence_index = None

    for position in range(4):
        if position == question_index:
            sentence_texts.append(training_question_sentence(target_city, target_season))
            sentence_vectors.append(training_sentence_vector("question", target_city, target_season))
        else:
            city, season, description = next(fact_iterator)
            fact = (city, season, description)
            if fact == correct_fact:
                answer_sentence_index = position
            sentence_texts.append(training_fact_sentence(city, season, description))
            sentence_vectors.append(training_sentence_vector("fact", city, season, description))

    return {
        "X": np.stack(sentence_vectors),
        "question_index": question_index,
        "target_index": TRAIN_DESCRIPTIONS.index(target_description),
        "answer_sentence_index": answer_sentence_index,
        "sentence_texts": sentence_texts,
        "answer": target_description,
    }

fixed_training_example = {
    "sentence_texts": sentences.copy(),
    "X": np.stack([
        training_sentence_vector("fact", "Zurich", "summer", "awesome"),
        training_sentence_vector("fact", "Bern", "summer", "cool"),
        training_sentence_vector("fact", "Lausanne", "summer", "nice view"),
        training_sentence_vector("question", "Zurich", "summer"),
    ]),
    "question_index": 3,
    "target_index": TRAIN_DESCRIPTIONS.index("awesome"),
    "answer_sentence_index": 0,
    "answer": "awesome",
}

class TinySentenceAttention(torch.nn.Module):
    """A minimal trainable sentence-level attention model."""

    def __init__(self, d_in, d_attention, n_answers):
        super().__init__()
        self.W_Q = torch.nn.Linear(d_in, d_attention, bias=False)
        self.W_K = torch.nn.Linear(d_in, d_attention, bias=False)
        self.W_V = torch.nn.Linear(d_in, d_attention, bias=False)
        self.classifier = torch.nn.Linear(d_attention, n_answers)

    def forward(self, sentence_batch, question_indices):
        queries = self.W_Q(sentence_batch)
        keys = self.W_K(sentence_batch)
        values = self.W_V(sentence_batch)

        batch_indices = torch.arange(sentence_batch.shape[0], device=sentence_batch.device)
        question_queries = queries[batch_indices, question_indices]
        scores = (
            keys * question_queries[:, None, :]
        ).sum(dim=-1) / math.sqrt(queries.shape[-1])

        question_mask = torch.nn.functional.one_hot(
            question_indices, num_classes=sentence_batch.shape[1]
        ).bool()
        scores = scores.masked_fill(question_mask, -1e9)

        attention = torch.softmax(scores, dim=-1)
        context = (attention[:, :, None] * values).sum(dim=1)
        logits = self.classifier(context)
        return logits, attention

def make_training_batch(batch_size):
    examples = [make_training_example() for _ in range(batch_size)]
    batch_X = torch.tensor(
        np.stack([example["X"] for example in examples]), dtype=torch.float32
    )
    question_indices = torch.tensor([
        example["question_index"] for example in examples
    ], dtype=torch.long)
    targets = torch.tensor([
        example["target_index"] for example in examples
    ], dtype=torch.long)
    answer_sentence_indices = torch.tensor([
        example["answer_sentence_index"] for example in examples
    ], dtype=torch.long)
    return batch_X, question_indices, targets, answer_sentence_indices

def run_training_example(model, example):
    model.eval()
    with torch.no_grad():
        example_X = torch.tensor(example["X"][None, :, :], dtype=torch.float32)
        question_index = torch.tensor([example["question_index"]], dtype=torch.long)
        logits, attention = model(example_X, question_index)
        probabilities = torch.softmax(logits, dim=-1)[0].cpu().numpy()
        prediction_index = int(probabilities.argmax())
    return {
        "attention": attention[0].cpu().numpy(),
        "probabilities": probabilities,
        "prediction": TRAIN_DESCRIPTIONS[prediction_index],
    }

def plot_training_prediction(example, result, title):
    bar_colors = []
    tick_labels = []
    for index in range(4):
        if index == example["question_index"]:
            bar_colors.append("#999999")
            role = "question"
        elif index == example["answer_sentence_index"]:
            bar_colors.append("#1b9e77")
            role = "answer fact"
        else:
            bar_colors.append("#d95f02")
            role = "distractor"
        tick_labels.append(f"S{index}\n{role}")

    fig, ax = plt.subplots(figsize=(9, 4.5))
    bars = ax.bar(tick_labels, result["attention"], color=bar_colors)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("attention weight from the question")
    ax.set_title(
        f"{title}\nprediction: {result['prediction']} | correct answer: {example['answer']}",
        loc="left",
    )
    for bar, value in zip(bars, result["attention"]):
        ax.text(bar.get_x() + bar.get_width() / 2, value + 0.025, f"{value:.2f}", ha="center")

    ax.legend(handles=[
        Patch(color="#1b9e77", label="answer fact"),
        Patch(color="#d95f02", label="distractor fact"),
        Patch(color="#999999", label="question (masked)"),
    ], loc="upper right")
    fig.tight_layout()
    plt.show()

for index, sentence in enumerate(fixed_training_example["sentence_texts"]):
    role = "question" if index == 3 else "answer fact" if index == 0 else "distractor fact"
    print(f"S{index}: {sentence:<42} ({role})")

training_model = TinySentenceAttention(
    TRAIN_D_IN, d_attention=16, n_answers=len(TRAIN_DESCRIPTIONS)
)
before_training = run_training_example(training_model, fixed_training_example)
plot_training_prediction(
    fixed_training_example,
    before_training,
    "Before training: random parameters",
)

#### 3.3.2 Training with gradient descent

Every training step generates a new batch of four-sentence examples. One distractor shares the requested city but uses another season; the other shares the requested season but uses another city. The correct fact may occur in any sentence position.

The loss uses only the correct description. The answer-sentence index is recorded solely to plot the green diagnostic curve; it is never used to update the model.

In [ ]:
optimizer = torch.optim.Adam(training_model.parameters(), lr=0.025)
training_steps = 220
training_batch_size = 96

training_history = {
    "step": [],
    "loss": [],
    "accuracy": [],
    "answer_attention": [],
}

for step in range(training_steps + 1):
    batch_X, batch_question_indices, batch_targets, batch_answer_indices = (
        make_training_batch(training_batch_size)
    )
    training_model.train()
    batch_logits, batch_attention = training_model(batch_X, batch_question_indices)
    loss = torch.nn.functional.cross_entropy(batch_logits, batch_targets)

    with torch.no_grad():
        predictions = batch_logits.argmax(dim=-1)
        accuracy = (predictions == batch_targets).float().mean().item()
        batch_indices = torch.arange(training_batch_size)
        answer_attention = batch_attention[
            batch_indices, batch_answer_indices
        ].mean().item()

    training_history["step"].append(step)
    training_history["loss"].append(float(loss.item()))
    training_history["accuracy"].append(accuracy)
    training_history["answer_attention"].append(answer_attention)

    if step < training_steps:
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8), constrained_layout=True)

axes[0].plot(training_history["step"], training_history["loss"], color="#4c78a8")
axes[0].set_title("Training loss", loc="left")
axes[0].set_xlabel("gradient descent step")
axes[0].set_ylabel("cross-entropy loss")

axes[1].plot(training_history["step"], training_history["accuracy"], color="#f58518")
axes[1].set_title("Batch accuracy", loc="left")
axes[1].set_xlabel("gradient descent step")
axes[1].set_ylabel("accuracy")
axes[1].set_ylim(0, 1.05)

axes[2].plot(
    training_history["step"],
    training_history["answer_attention"],
    color="#1b9e77",
)
axes[2].set_title("Attention on the answer fact", loc="left")
axes[2].set_xlabel("gradient descent step")
axes[2].set_ylabel("mean attention weight")
axes[2].set_ylim(0, 1.05)

plt.show()

#### 3.3.3 Prediction after training

We now give the trained model exactly the same Zurich prompt used before training. If learning worked, the green Zurich fact should receive most of the question's attention and the predicted answer should be `awesome`.

In [ ]:
after_training = run_training_example(training_model, fixed_training_example)
plot_training_prediction(
    fixed_training_example,
    after_training,
    "After training: attention supports the answer",
)

evaluation_X, evaluation_questions, evaluation_targets, _ = make_training_batch(1000)
training_model.eval()
with torch.no_grad():
    evaluation_logits, _ = training_model(evaluation_X, evaluation_questions)
    evaluation_accuracy = (
        evaluation_logits.argmax(dim=-1) == evaluation_targets
    ).float().mean().item()

print("Prediction before training:", before_training["prediction"])
print("Prediction after training: ", after_training["prediction"])
print("Correct answer:            ", fixed_training_example["answer"])
print(f"Accuracy on 1,000 new prompts: {evaluation_accuracy:.1%}")

The comparison shows what gradient descent contributed. Before training, the attention pattern came from random matrices and was not a meaningful routing rule. During training, loss decreased while both answer accuracy and attention on the relevant fact increased. After training, the question routed most of its information from the Zurich fact and the model predicted `awesome`.

No attention target was supplied during optimization. The useful attention pattern emerged because it helped the model reduce its answer-prediction loss.

### 3.4 Compute Attention Scores

Attention scores are computed by multiplying queries with keys:

`scores = Q @ K.T / sqrt(d_k)`

The result is a square matrix with one row per sentence and one column per sentence. Entry `(i, j)` says how strongly sentence `i` attends to sentence `j` before normalization.

The division by `sqrt(d_k)` keeps dot products from growing too large as the projection dimension increases. Without this scaling, softmax can become too sharp early in training.

In [ ]:
scores = (Q @ K.T) / np.sqrt(d_k)

print("Attention score matrix:")
print(scores)
print("Shape:", scores.shape)

for i, label in enumerate(sentence_labels):
    print(f"{label!r} raw scores against {sentence_labels}: {scores[i]}")

### 3.5 Turn Scores Into Weights

Raw attention scores are not yet probabilities. Softmax converts each row into non-negative weights that sum to one.

After softmax, every row tells us how much one sentence will pull information from each sentence's value vector.

This is the moment where attention becomes a weighted averaging operation: high-weight values influence the output more strongly, and low-weight values contribute less.

In [ ]:
def softmax_rows(A):
    shifted = A - A.max(axis=1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=1, keepdims=True)

attention_weights = softmax_rows(scores)

print("Attention weights:")
print(attention_weights)
print("Row sums:", attention_weights.sum(axis=1))

for i, label in enumerate(sentence_labels):
    weights = {
        source: float(np.round(weight, 3))
        for source, weight in zip(sentence_labels, attention_weights[i])
    }
    print(f"{label!r} attends with weights: {weights}")

### 3.6 Combine the Values

The final self-attention output is:

`output = attention_weights @ V`

Each output row is a weighted combination of all value vectors. This gives every sentence a new contextual representation that depends on the other sentences in the sequence.

The key point: Q and K choose the mixing weights; V supplies the information being mixed.

In [ ]:
context = attention_weights @ V

print("Contextualized output:")
print(context)
print("Shape:", context.shape)

value_features = ["awesome", "cool", "nice view"]
for label, vec in zip(sentence_labels, context):
    print(f"{label!r} new representation: {np.round(vec, 3)}")

question_index = 3
question_result = {
    feature: float(np.round(value, 3))
    for feature, value in zip(value_features, context[question_index])
}
print("\nDescription retrieved for the Zurich question:", question_result)

### 3.7 Visualize Attention

A heatmap makes the attention matrix easier to inspect. Rows show the sentence doing the attending, and columns show the sentence being attended to. The final row is the most important one for this example: it shows where the Zurich question retrieves its information.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
image = ax.imshow(attention_weights, cmap="Blues", vmin=0, vmax=1)

ax.set_xticks(range(len(sentence_labels)), sentence_labels, rotation=25, ha="right")
ax.set_yticks(range(len(sentence_labels)), sentence_labels)
ax.set_xlabel("sentence being attended to")
ax.set_ylabel("query sentence")
ax.set_title("Sentence-level attention weights")

for row in range(len(sentence_labels)):
    for column in range(len(sentence_labels)):
        weight = attention_weights[row, column]
        text_color = "white" if weight > 0.5 else "black"
        ax.text(column, row, f"{weight:.3f}", ha="center", va="center", color=text_color)

fig.colorbar(image, ax=ax, label="attention weight")
fig.tight_layout()
plt.show()

---

### 3.8 Questions

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>

**Question 01**

What are the roles of `Q`, `K`, and `V` in attention?

Hint: `Q` and `K` decide where attention goes; `V` carries the information that gets mixed.

Answer:

 - (add your short answer here)

**Question 02**

In the fixed NumPy example from Section 3.1, the input matrix `X` has shape `(4, 8)`. What do the two dimensions represent?

Hint: There are four sentences, and each sentence is represented by eight human-readable features.

Answer:

 - (add your short answer here)

**Question 03**

Why does the notebook compute `Q = X @ W_Q`, `K = X @ W_K`, and `V = X @ W_V` instead of using `X` directly for every role?

Hint: The projection matrices create separate spaces for matching sentences and carrying descriptive content.

Answer:

 - (add your short answer here)

**Question 04**

What does the matrix product `Q @ K.T` measure before softmax is applied?

Hint: Each row compares one query sentence against every key sentence using dot products.

Answer:

 - (add your short answer here)

**Question 05**

Why are the attention scores divided by `sqrt(d_k)`?

Hint: Scaling keeps dot products from becoming too large as the key/query dimension grows, which makes softmax less extreme.

Answer:

 - (add your short answer here)

**Question 06**

After softmax, why does each row of the attention-weight matrix sum to `1`?

Hint: Each row is a distribution over which sentences one query sentence attends to.

Answer:

 - (add your short answer here)

**Question 07**

What does `context = attention_weights @ V` compute?

Hint: Each output row is a weighted sum of the value vectors.

Answer:

 - (add your short answer here)

**Question 08**

Which factual sentence should the Zurich question attend to most strongly, and why?

Hint: One fact matches both the city `Zurich` and the season `summer`; the other facts match only the season.

Answer:

 - (add your short answer here)

**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

 (please help the team by telling us what part was not understood, and we will work on it)